# Phase 1 Notebook — Retrieval Basics

This notebook is aligned with the Phase 1 instructions:
- load the dataset with the `json` module;
- inspect fields and compute dataset statistics;
- build a shared `content` field for documents and queries;
- implement the three required retrieval methods: TF-IDF, BM25+, and embeddings;
- evaluate them with `Precision`, `Recall`, and `MRR`;
- inspect embeddings in 2D;
- compare models and produce the Kaggle submission.

Besides the required metrics, the notebook also computes `MAP@k` as an additional ranking metric and reports a local `FinalScore@k = 0.25 * (Precision@k + Recall@k + MRR@k + MAP@k)` to make model comparison easier.

In [ ]:
from pathlib import Path
import csv
import json
import os
import re
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.manifold import TSNE
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option('display.max_colwidth', 140)

TOP_K = 100
FINAL_MODEL = 'embedding'
EMBEDDING_MODEL = 'all-MiniLM-L6-v2'
EMBEDDING_BATCH = 256
TSNE_SAMPLE_SIZE = 2000
RANDOM_STATE = 42
OUTPUT_PATH = Path('solutions_SeaFour.csv')
PROJECT_FOLDER_NAME = 'retrieval_project'

# Keep TOP_K=100 for the default submission.
# For the report, study the influence of k later with values such as [10, 20, 50, 100].

TOKEN_RE = re.compile(r'[a-z0-9]+')


def is_colab():
    try:
        import google.colab  # type: ignore
        return True
    except ImportError:
        return False


def maybe_mount_google_drive():
    try:
        from google.colab import drive  # type: ignore
    except ImportError:
        return

    drive_root = Path('/content/drive/MyDrive')
    if drive_root.exists():
        print('Google Drive already mounted.')
        return

    print('Google Colab detected. Mounting Google Drive...')
    drive.mount('/content/drive')


def candidate_data_dirs():
    cwd = Path.cwd().resolve()
    candidates = [
        Path('/kaggle/input/retrieval-engine-competition'),
        cwd / 'data',
        cwd.parent / 'data',
        cwd.parent.parent / 'data',
    ]

    for parent in (cwd, *cwd.parents):
        if parent.name == PROJECT_FOLDER_NAME:
            candidates.append(parent / 'data')

    drive_root = Path('/content/drive/MyDrive')
    if drive_root.exists():
        candidates.append(drive_root / PROJECT_FOLDER_NAME / 'data')
        candidates.append(drive_root / 'Colab Notebooks' / PROJECT_FOLDER_NAME / 'data')
        candidates.extend(path / 'data' for path in drive_root.glob(f'*/{PROJECT_FOLDER_NAME}'))

    unique_candidates = []
    seen = set()
    for candidate in candidates:
        candidate = candidate.expanduser()
        if candidate in seen:
            continue
        seen.add(candidate)
        unique_candidates.append(candidate)
    return unique_candidates


def resolve_data_dir():
    maybe_mount_google_drive()
    cwd = Path.cwd().resolve()
    print(f'Current working directory: {cwd}')
    print(f'Running in Colab: {is_colab()}')
    print('Searching for dataset directory...')

    for dirname, _, filenames in os.walk('/kaggle/input'):
        if ('docs.json' in filenames or 'documents.json' in filenames) and 'queries_test.json' in filenames:
            print(f'Using Kaggle dataset directory: {dirname}')
            return Path(dirname)

    searched = []
    for candidate in candidate_data_dirs():
        print(f'Checking data directory candidate: {candidate}')
        searched.append(str(candidate))
        has_docs = (candidate / 'docs.json').exists() or (candidate / 'documents.json').exists()
        has_queries = (candidate / 'queries_test.json').exists()
        if has_docs and has_queries:
            print(f'Using dataset directory: {candidate}')
            return candidate

    raise FileNotFoundError(
        'Could not find the dataset directory. Checked:\n'
        + '\n'.join(searched)
        + '\n\nExpected docs.json or documents.json and queries_test.json under the data/ directory.'
    )


def existing_path(data_dir, names):
    for name in names:
        candidate = data_dir / name
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f'Could not find any of: {names}')


def load_json_records(path):
    with open(path, 'r', encoding='utf-8') as f:
        raw = json.load(f)
    if isinstance(raw, list):
        return raw
    if isinstance(raw, dict):
        for key in ['documents', 'docs', 'queries', 'items', 'data']:
            value = raw.get(key)
            if isinstance(value, list):
                return value
        if all(isinstance(v, dict) for v in raw.values()):
            return list(raw.values())
    raise ValueError(f'Unsupported JSON structure in {path}')


def value_to_text(value):
    if value is None:
        return ''
    if isinstance(value, (list, tuple)):
        return ' '.join(str(v) for v in value)
    if pd.isna(value):
        return ''
    return str(value)


def create_content_column(df, columns):
    out = df.copy()
    for col in columns:
        if col not in out.columns:
            out[col] = ''
    merged = []
    for _, row in out[columns].iterrows():
        merged.append(' '.join(value_to_text(row[col]) for col in columns).strip().lower())
    out['content'] = merged
    out['id'] = out['id'].astype(str)
    return out


def tokenize(text):
    txt = str(text or '').lower()
    txt = re.sub(r'[-_/]', ' ', txt)
    return TOKEN_RE.findall(txt)


def token_length_stats(texts):
    lengths = texts.fillna('').map(lambda text: len(tokenize(text)))
    return {
        'min': int(lengths.min()),
        'max': int(lengths.max()),
        'mean': float(lengths.mean()),
        'median': float(lengths.median()),
        'std': float(lengths.std(ddof=0)),
    }


def load_ground_truth(path):
    with open(path, 'r', encoding='utf-8') as f:
        raw = json.load(f)

    ground_truth = {}
    total_judgments = 0

    for qid, info in raw.items():
        if isinstance(info, dict):
            items = info.get('relevant_doc_ids', [])
        else:
            items = info

        relevant = set()
        for item in items:
            total_judgments += 1
            if isinstance(item, dict):
                doc_id = item.get('doc_id', item.get('id'))
                relevance = item.get('relevance', item.get('label', 1))
            else:
                doc_id = item
                relevance = 1
            if relevance and doc_id is not None:
                relevant.add(str(doc_id))
        ground_truth[str(qid)] = relevant

    return ground_truth, raw, total_judgments


def describe_available_fields(df):
    rows = []
    for col in df.columns:
        example = value_to_text(df.iloc[0][col]) if len(df) else ''
        rows.append({
            'field': col,
            'dtype': str(df[col].dtype),
            'non_null': int(df[col].notna().sum()),
            'sample': example[:120],
        })
    return pd.DataFrame(rows)


def compute_dataset_statistics(docs_df, train_queries_df, test_queries_df, ground_truth, total_judgments):
    rel_counts = pd.Series([len(ground_truth.get(str(qid), set())) for qid in train_queries_df['id']], dtype='int64')
    category_counts = docs_df['category'].fillna('unknown').astype(str).value_counts().sort_values(ascending=False) if 'category' in docs_df.columns else pd.Series(dtype='int64')
    return {
        'num_documents': int(len(docs_df)),
        'num_train_queries': int(len(train_queries_df)),
        'num_test_queries': int(len(test_queries_df)),
        'num_relevance_judgments': int(total_judgments),
        'num_categories': int(len(category_counts)),
        'category_counts': category_counts,
        'doc_length_stats': token_length_stats(docs_df['content']),
        'train_query_length_stats': token_length_stats(train_queries_df['content']),
        'test_query_length_stats': token_length_stats(test_queries_df['content']),
        'relevant_docs_per_query': {
            'min': int(rel_counts.min()),
            'max': int(rel_counts.max()),
            'mean': float(rel_counts.mean()),
            'median': float(rel_counts.median()),
            'std': float(rel_counts.std(ddof=0)),
        },
    }


def topk_from_scores(scores, top_k):
    top_k = min(top_k, scores.shape[1])
    part = np.argpartition(scores, -top_k, axis=1)[:, -top_k:]
    part_scores = np.take_along_axis(scores, part, axis=1)
    order = np.argsort(part_scores, axis=1)[:, ::-1]
    topk_indices = np.take_along_axis(part, order, axis=1)
    topk_scores = np.take_along_axis(scores, topk_indices, axis=1)
    return topk_indices, topk_scores


def build_run(query_ids, doc_ids, topk_indices, topk_scores, latency_sec):
    return {
        'query_ids': np.asarray(query_ids, dtype=str),
        'doc_ids': np.asarray(doc_ids, dtype=str),
        'topk_indices': topk_indices,
        'topk_scores': topk_scores,
        'topk_doc_ids': np.asarray(doc_ids, dtype=str)[topk_indices],
        'latency_sec': float(latency_sec),
    }


def run_tfidf_search(docs_df, queries_df, top_k=100):
    t0 = time.time()
    vectorizer = TfidfVectorizer(lowercase=True, ngram_range=(1, 2), min_df=2)
    try:
        doc_vectors = vectorizer.fit_transform(docs_df['content'])
    except ValueError as err:
        if 'After pruning, no terms remain' not in str(err):
            raise
        vectorizer = TfidfVectorizer(lowercase=True, ngram_range=(1, 2), min_df=1)
        doc_vectors = vectorizer.fit_transform(docs_df['content'])
    query_vectors = vectorizer.transform(queries_df['content'])
    scores = cosine_similarity(query_vectors, doc_vectors)
    topk_indices, topk_scores = topk_from_scores(scores, top_k)
    return build_run(queries_df['id'], docs_df['id'], topk_indices, topk_scores, time.time() - t0)


def run_bm25_search(docs_df, queries_df, top_k=100):
    from rank_bm25 import BM25Plus
    t0 = time.time()
    bm25 = BM25Plus([tokenize(text) for text in docs_df['content']])
    scores = np.vstack([bm25.get_scores(tokenize(text)) for text in queries_df['content']])
    topk_indices, topk_scores = topk_from_scores(scores, top_k)
    return build_run(queries_df['id'], docs_df['id'], topk_indices, topk_scores, time.time() - t0)


def encode_embeddings(docs_df, queries_df, model_name=EMBEDDING_MODEL, batch_size=EMBEDDING_BATCH):
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer(model_name)
    doc_embeddings = model.encode(docs_df['content'].tolist(), batch_size=batch_size, show_progress_bar=True, normalize_embeddings=True)
    query_embeddings = model.encode(queries_df['content'].tolist(), batch_size=batch_size, show_progress_bar=True, normalize_embeddings=True)
    return np.asarray(doc_embeddings), np.asarray(query_embeddings)


def run_embedding_search(docs_df, queries_df, top_k=100, model_name=EMBEDDING_MODEL, batch_size=EMBEDDING_BATCH):
    t0 = time.time()
    doc_embeddings, query_embeddings = encode_embeddings(docs_df, queries_df, model_name=model_name, batch_size=batch_size)
    scores = query_embeddings @ doc_embeddings.T
    topk_indices, topk_scores = topk_from_scores(scores, top_k)
    run = build_run(queries_df['id'], docs_df['id'], topk_indices, topk_scores, time.time() - t0)
    return run, doc_embeddings, query_embeddings


def precision_at_k(run, ground_truth, k=None):
    doc_ids = run['topk_doc_ids']
    query_ids = run['query_ids']
    k = doc_ids.shape[1] if k is None else k
    values = []
    for idx, qid in enumerate(query_ids):
        relevant = ground_truth.get(str(qid), set())
        retrieved = [str(doc_id) for doc_id in doc_ids[idx][:k]]
        hits = sum(1 for doc_id in retrieved if doc_id in relevant)
        values.append(hits / len(retrieved) if retrieved else 0.0)
    return float(np.mean(values)) if values else 0.0


def recall_at_k(run, ground_truth, k=None):
    doc_ids = run['topk_doc_ids']
    query_ids = run['query_ids']
    k = doc_ids.shape[1] if k is None else k
    values = []
    for idx, qid in enumerate(query_ids):
        relevant = ground_truth.get(str(qid), set())
        if not relevant:
            values.append(0.0)
            continue
        retrieved = {str(doc_id) for doc_id in doc_ids[idx][:k]}
        values.append(len(relevant & retrieved) / len(relevant))
    return float(np.mean(values)) if values else 0.0


def mrr_at_k(run, ground_truth, k=None):
    doc_ids = run['topk_doc_ids']
    query_ids = run['query_ids']
    k = doc_ids.shape[1] if k is None else k
    values = []
    for idx, qid in enumerate(query_ids):
        relevant = ground_truth.get(str(qid), set())
        rr = 0.0
        for rank, doc_id in enumerate(doc_ids[idx][:k], start=1):
            if str(doc_id) in relevant:
                rr = 1.0 / rank
                break
        values.append(rr)
    return float(np.mean(values)) if values else 0.0


def map_at_k(run, ground_truth, k=None):
    doc_ids = run['topk_doc_ids']
    query_ids = run['query_ids']
    k = doc_ids.shape[1] if k is None else k
    values = []
    for idx, qid in enumerate(query_ids):
        relevant = ground_truth.get(str(qid), set())
        if not relevant:
            values.append(0.0)
            continue
        hits = 0
        ap = 0.0
        for rank, doc_id in enumerate(doc_ids[idx][:k], start=1):
            if str(doc_id) in relevant:
                hits += 1
                ap += hits / rank
        values.append(ap / len(relevant))
    return float(np.mean(values)) if values else 0.0


def evaluate_run(run, ground_truth, k):
    precision = precision_at_k(run, ground_truth, k=k)
    recall = recall_at_k(run, ground_truth, k=k)
    mrr = mrr_at_k(run, ground_truth, k=k)
    mean_ap = map_at_k(run, ground_truth, k=k)
    final_score = 0.25 * (precision + recall + mrr + mean_ap)
    return {
        f'Precision@{k}': precision,
        f'Recall@{k}': recall,
        f'MRR@{k}': mrr,
        f'MAP@{k}': mean_ap,
        f'FinalScore@{k}': final_score,
        'LatencySec': float(run['latency_sec']),
    }


def sample_indices_for_tsne(docs_df, sample_size=TSNE_SAMPLE_SIZE, random_state=RANDOM_STATE):
    sample_size = min(sample_size, len(docs_df))
    rng = np.random.default_rng(random_state)
    if 'category' not in docs_df.columns or docs_df['category'].isna().all():
        return np.sort(rng.choice(len(docs_df), size=sample_size, replace=False))
    groups = docs_df['category'].fillna('unknown').astype(str)
    categories = groups.unique().tolist()
    per_category = max(1, sample_size // max(1, len(categories)))
    sampled = []
    for category in categories:
        idx = np.flatnonzero(groups.to_numpy() == category)
        take = min(per_category, len(idx))
        if take:
            sampled.extend(rng.choice(idx, size=take, replace=False).tolist())
    sampled = sorted(set(sampled))
    if len(sampled) < sample_size:
        remaining = np.setdiff1d(np.arange(len(docs_df)), np.asarray(sampled, dtype=int), assume_unique=False)
        extra = min(sample_size - len(sampled), len(remaining))
        if extra:
            sampled.extend(rng.choice(remaining, size=extra, replace=False).tolist())
    return np.asarray(sorted(sampled[:sample_size]), dtype=int)


def plot_embedding_projection(docs_df, doc_embeddings, query_embeddings, sample_size=TSNE_SAMPLE_SIZE, random_state=RANDOM_STATE):
    sampled_doc_indices = sample_indices_for_tsne(docs_df, sample_size=sample_size, random_state=random_state)
    combined = np.vstack([doc_embeddings[sampled_doc_indices], query_embeddings])
    projection = TSNE(n_components=2, perplexity=30, random_state=random_state, init='pca', learning_rate='auto').fit_transform(combined)
    doc_projection = projection[:len(sampled_doc_indices)]
    query_projection = projection[len(sampled_doc_indices):]

    plt.figure(figsize=(10, 7))
    if 'category' in docs_df.columns:
        categories = docs_df.iloc[sampled_doc_indices]['category'].fillna('unknown').astype(str)
        for category in sorted(categories.unique()):
            mask = (categories == category).to_numpy()
            plt.scatter(doc_projection[mask, 0], doc_projection[mask, 1], s=12, alpha=0.55, label=f'doc:{category}')
    else:
        plt.scatter(doc_projection[:, 0], doc_projection[:, 1], s=12, alpha=0.5, label='documents')
    plt.scatter(query_projection[:, 0], query_projection[:, 1], s=40, c='black', marker='x', linewidths=1.0, label='train queries')
    plt.title('t-SNE projection of document and query embeddings')
    plt.xlabel('t-SNE dim 1')
    plt.ylabel('t-SNE dim 2')
    plt.legend(loc='best', fontsize=8)
    plt.tight_layout()
    return doc_projection, query_projection


def write_kaggle_submission(run, sample_csv_path, output_csv_path):
    pred_map = {str(qid): [str(doc_id) for doc_id in run['topk_doc_ids'][idx].tolist()] for idx, qid in enumerate(run['query_ids'])}
    with open(sample_csv_path, 'r', newline='', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        fieldnames = reader.fieldnames
        rows = list(reader)
    if fieldnames is None or len(fieldnames) < 2:
        raise ValueError('Invalid sample submission format.')
    id_col = fieldnames[0]
    pred_col = fieldnames[1]
    category_col = fieldnames[2] if len(fieldnames) >= 3 else None
    with open(output_csv_path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for row in rows:
            qid = str(row[id_col])
            if qid not in pred_map:
                raise ValueError(f'Missing prediction for query_id: {qid}')
            out_row = {id_col: qid, pred_col: json.dumps(pred_map[qid])}
            if category_col is not None:
                out_row[category_col] = row.get(category_col, '?') or '?'
            writer.writerow(out_row)


DATA_DIR = resolve_data_dir()
DOCS_PATH = existing_path(DATA_DIR, ['docs.json', 'documents.json'])
TRAIN_QUERIES_PATH = existing_path(DATA_DIR, ['queries_train.json'])
TEST_QUERIES_PATH = existing_path(DATA_DIR, ['queries_test.json'])
QRELS_PATH = existing_path(DATA_DIR, ['qgts_train.json', 'gts.json'])
SUBMISSION_TEMPLATE_PATH = existing_path(DATA_DIR, ['submission.csv'])

print(f'Data directory: {DATA_DIR}')
print(f'Final submission model: {FINAL_MODEL}')

## Load And Prepare The Data

The project instructions explicitly ask to load the dataset with the `json` module. The raw JSON records are loaded below and then converted to pandas DataFrames for easier analysis.

The `content` field is created by concatenating the available `title`, `text`, and `tags` fields. This gives each retrieval model the same textual representation.

In [ ]:
docs_raw = load_json_records(DOCS_PATH)
train_queries_raw = load_json_records(TRAIN_QUERIES_PATH)
test_queries_raw = load_json_records(TEST_QUERIES_PATH)
ground_truth, qrels_raw, total_judgments = load_ground_truth(QRELS_PATH)

docs_df = pd.DataFrame(docs_raw)
train_queries_df = pd.DataFrame(train_queries_raw)
test_queries_df = pd.DataFrame(test_queries_raw)

docs_df = create_content_column(docs_df, ['title', 'text', 'tags'])
train_queries_df = create_content_column(train_queries_df, ['title', 'text', 'tags'])
test_queries_df = create_content_column(test_queries_df, ['title', 'text', 'tags'])

print(f'Documents             : {len(docs_df):,}')
print(f'Train queries         : {len(train_queries_df):,}')
print(f'Test queries          : {len(test_queries_df):,}')
print(f'Relevance judgments   : {total_judgments:,}')

display(docs_df.head(2))
display(train_queries_df.head(2))

## Dataset Inspection

This section addresses the first part of Phase 1: understanding what the dataset looks like, which fields it contains, how long documents and queries are, how many categories exist, and how many relevant documents appear per training query.

In [ ]:
doc_fields_df = describe_available_fields(docs_df)
train_query_fields_df = describe_available_fields(train_queries_df)
dataset_stats = compute_dataset_statistics(docs_df, train_queries_df, test_queries_df, ground_truth, total_judgments)

summary_df = pd.DataFrame([
    {'statistic': 'num_documents', 'value': dataset_stats['num_documents']},
    {'statistic': 'num_train_queries', 'value': dataset_stats['num_train_queries']},
    {'statistic': 'num_test_queries', 'value': dataset_stats['num_test_queries']},
    {'statistic': 'num_relevance_judgments', 'value': dataset_stats['num_relevance_judgments']},
    {'statistic': 'num_categories', 'value': dataset_stats['num_categories']},
])

length_stats_df = pd.DataFrame([
    {'group': 'documents', **dataset_stats['doc_length_stats']},
    {'group': 'train_queries', **dataset_stats['train_query_length_stats']},
    {'group': 'test_queries', **dataset_stats['test_query_length_stats']},
    {'group': 'relevant_docs_per_query', **dataset_stats['relevant_docs_per_query']},
])

print('Document fields')
display(doc_fields_df)
print('Train query fields')
display(train_query_fields_df)
print('High-level statistics')
display(summary_df)
print('Length and relevance statistics')
display(length_stats_df)

if dataset_stats['num_categories']:
    category_counts_df = dataset_stats['category_counts'].rename_axis('category').reset_index(name='count')
    print('Document categories')
    display(category_counts_df)

## Dataset Conclusions

Use this section in the final report to explain what the statistics suggest. Typical points to discuss are:
- whether documents are much longer than queries;
- whether relevance labels are sparse or dense;
- whether some categories dominate the corpus;
- why these properties may help lexical models or semantic embeddings.

## TF-IDF Retrieval

TF-IDF stands for Term Frequency–Inverse Document Frequency. It gives higher weight to terms that occur frequently inside a document but are relatively rare in the whole corpus. This helps highlight words that are informative for that document. After vectorizing documents and queries, cosine similarity is used to rank the documents for each query.

The required outputs for this method are `topk_indices_tfidf` and `topk_scores_tfidf`.

## BM25+ Retrieval

BM25+ is a probabilistic retrieval model designed for ranking documents. Compared with plain TF-IDF, it includes term-frequency saturation and document-length normalization, so repeated occurrences of a term help only up to a point and long documents are not unfairly favored. BM25+ is often a strong lexical baseline for retrieval.

The required outputs for this method are `topk_indices_bm25` and `topk_scores_bm25`.

## Representation Learning With Embeddings

High-dimensional embeddings convert text into dense vectors that capture semantic meaning. Their purpose is to place texts with similar meaning close to each other in vector space, even if they do not share the same exact words. This helps information retrieval because embeddings can handle synonyms, paraphrases, and related concepts better than purely lexical methods.

The required outputs for this method are `topk_indices_embedding` and `topk_scores_embedding`.

In [ ]:
!pip install rank_bm25

In [ ]:
tfidf_run = run_tfidf_search(docs_df, train_queries_df, top_k=TOP_K)
bm25_run = run_bm25_search(docs_df, train_queries_df, top_k=TOP_K)
embedding_run, doc_embeddings, query_embeddings = run_embedding_search(docs_df, train_queries_df, top_k=TOP_K, model_name=EMBEDDING_MODEL, batch_size=EMBEDDING_BATCH)

topk_indices_tfidf = tfidf_run['topk_indices']
topk_scores_tfidf = tfidf_run['topk_scores']

topk_indices_bm25 = bm25_run['topk_indices']
topk_scores_bm25 = bm25_run['topk_scores']

topk_indices_embedding = embedding_run['topk_indices']
topk_scores_embedding = embedding_run['topk_scores']

print(f'topk_indices_tfidf shape     : {topk_indices_tfidf.shape}')
print(f'topk_scores_tfidf shape      : {topk_scores_tfidf.shape}')
print(f'topk_indices_bm25 shape      : {topk_indices_bm25.shape}')
print(f'topk_scores_bm25 shape       : {topk_scores_bm25.shape}')
print(f'topk_indices_embedding shape : {topk_indices_embedding.shape}')
print(f'topk_scores_embedding shape  : {topk_scores_embedding.shape}')

## Embedding Inspection

The assignment asks to inspect the structure of embeddings and visualize them in 2D. The code below prints the embedding shapes and draws a t-SNE projection. If categories are available, document points are colored by category to make clustering easier to interpret.

In [ ]:
print(f'Document embeddings shape: {doc_embeddings.shape}')
print(f'Query embeddings shape   : {query_embeddings.shape}')

plot_embedding_projection(docs_df, doc_embeddings, query_embeddings)

## Embedding Interpretation

After running the t-SNE projection, describe whether you observe clustering or separation. A reasonable explanation is that documents from similar categories or topics tend to be embedded closer together, while queries often lie near documents that share semantic meaning.

## Retrieval Evaluation Suite

The required evaluation metrics are:
- `Precision@k`
- `Recall@k`
- `MRR@k`

The notebook also reports `MAP@k` as an additional ranking metric and computes `FinalScore@k = 0.25 * (Precision@k + Recall@k + MRR@k + MAP@k)`.

This final score is not a replacement for the required metrics; it is a local summary score so that each metric has weight `0.25`.

In [ ]:
evaluation_rows = {
    'tfidf': evaluate_run(tfidf_run, ground_truth, k=TOP_K),
    'bm25': evaluate_run(bm25_run, ground_truth, k=TOP_K),
    'embedding': evaluate_run(embedding_run, ground_truth, k=TOP_K),
}

eval_df = pd.DataFrame(evaluation_rows).T
eval_df.index.name = 'Model'
eval_df = eval_df.sort_values(by=f'FinalScore@{TOP_K}', ascending=False)
display(eval_df)

## Compare The Retrieval Models

Use the table above to answer the Phase 1 comparison questions in your report:
- Which model has the best precision, recall, and MRR?
- Why might lexical models do well on some queries and embeddings on others?
- What are the trade-offs between retrieval quality and latency?
- How does the value of `k` affect each metric?

Latency is already included in the table to support the speed comparison.

## Influence Of `k`

The instructions explicitly say that `k` should be experimented with. The code below is intentionally left as comments so you can run your own sensitivity study and discuss how `Precision`, `Recall`, `MRR`, `MAP`, and `FinalScore` evolve with different values of `k`.

In [ ]:
# Example template for a manual k-study:
#
# K_VALUES = [10, 20, 50, 100]
# k_rows = []
# for k in K_VALUES:
#     tfidf_k = run_tfidf_search(docs_df, train_queries_df, top_k=k)
#     bm25_k = run_bm25_search(docs_df, train_queries_df, top_k=k)
#     embedding_k, _, _ = run_embedding_search(docs_df, train_queries_df, top_k=k, model_name=EMBEDDING_MODEL, batch_size=EMBEDDING_BATCH)
#
#     for model_name, run in [('tfidf', tfidf_k), ('bm25', bm25_k), ('embedding', embedding_k)]:
#         row = evaluate_run(run, ground_truth, k=k)
#         row['Model'] = model_name
#         row['k'] = k
#         k_rows.append(row)
#
# k_study_df = pd.DataFrame(k_rows)
# display(k_study_df)

## Final Kaggle Submission

The submission cell below keeps `embedding` as the final submission model, while TF-IDF and BM25+ remain in the notebook for the required comparison.

In [ ]:
if FINAL_MODEL != 'embedding':
    raise ValueError('This notebook is configured to generate the final submission with the embedding model.')

test_embedding_run, _, _ = run_embedding_search(docs_df, test_queries_df, top_k=TOP_K, model_name=EMBEDDING_MODEL, batch_size=EMBEDDING_BATCH)
write_kaggle_submission(test_embedding_run, SUBMISSION_TEMPLATE_PATH, OUTPUT_PATH)
print(f'Saved: {OUTPUT_PATH.resolve()}')

In [ ]:
submission_preview = pd.read_csv(OUTPUT_PATH)
print(f'Rows: {len(submission_preview)}, Columns: {list(submission_preview.columns)}')
submission_preview.head()